# Capstone build --- Chapter 7: Cost and Latency Budgets

A loop that runs until the agent chooses to finish can run without bound if the agent never does. Chapter~7 puts a ceiling on the loop. A `Budget` declares per-axis upper bounds --- tokens, seconds, tool calls, dollars --- and a `BudgetTracker` accumulates consumption against them. The loop checks the tracker before each step and stops with an escalation once any axis is exhausted, so a runaway or wedged case ends in a bounded, auditable way.

## Declaring a budget

Each axis of a `Budget` is an upper bound, and `None` means unlimited on that axis. The complaint agent bounds the number of tool calls, which caps a case at its fixed five-step workflow plus a margin; it leaves tokens and dollars unbounded because the local models carry no per-call charge.

In [ ]:
from forgeloop.agents.core import Budget, BudgetTracker

budget = Budget(tool_calls=20)
tracker = BudgetTracker(budget)
print('bounds     :', budget)
print('exhausted? :', tracker.exhausted())

## Accumulating against the bound

The tracker records consumption as the loop runs. Recording tool calls up to the bound leaves the tracker exhausted, and it reports which axis ran out. This is the check `run_loop` performs before each step.

In [ ]:
small = BudgetTracker(Budget(tool_calls=3))
for i in range(3):
    small.record_tool_call()
    print(f'after {i+1} calls: exhausted={small.exhausted()}')
print('reason      :', small.reason_exhausted())
print('consumption :', small.consumption())

## The loop stops on an exhausted budget

`run_loop` accepts a `budget_tracker`. Before proposing an action it checks whether the budget is exhausted; if so it yields a final `Escalate` step with the exhaustion reason and stops. Starting the loop with an already-exhausted tracker shows the mechanism without needing to run a tool: the very first step is the budget escalation.

In [ ]:
from forgeloop.agents.core.agent import BaseAgent
from forgeloop.agents.core.action import Finish
from forgeloop.agents.core.state import AgentState
from forgeloop.agents.core.task import TaskSpec
from forgeloop.agents.core.loop import run_loop

class NeverFinishes(BaseAgent):
    def propose_action(self, state):
        return Finish(output=None)  # never reached: the budget check fires first
    def update(self, state, action, observation):
        return state

state = AgentState(task=TaskSpec(goal='demo', inputs={}))
exhausted = BudgetTracker(Budget(tool_calls=0))  # nothing allowed
for rec in run_loop(NeverFinishes(), None, state, max_steps=4, budget_tracker=exhausted):
    print(f'step {rec.step}: {rec.action.kind} -> {rec.state_after.status}')
    if rec.action.kind == 'escalate':
        print('   reason:', rec.action.reason)

The budget makes termination a property of the harness, not a hope about the agent. In the capstone every case runs under `BudgetTracker(Budget(tool_calls=20))`, so a case that fails to converge escalates on the budget axis and is logged like any other escalation. Chapter~8 gives the agent the fixed plan that keeps a well-behaved case well inside this bound.